### Step 1. Prepare calibration data.

In [31]:
import numpy as np
from onnxruntime.quantization import CalibrationDataReader, quantize_static, QuantType, QuantFormat
import cv2
	
class ImageCalibrationDataReader(CalibrationDataReader):
	def __init__(self, image_paths):
		self.image_paths = image_paths
		self.idx = 0
		self.input_name = "images"

	def preprocess(self, frame):
		frame = cv2.imread(frame)
		X = cv2.resize(frame, (640, 640))
		image_data = np.array(X).astype(np.float32) / 255.0  # Normalize to [0, 1] range
		image_data = np.transpose(image_data, (2, 0, 1))  # (H, W, C) -> (C, H, W)
		image_data = np.expand_dims(image_data, axis=0)  # Add batch dimension
		return image_data

	def get_next(self):
		if self.idx >= len(self.image_paths):
			return None

		batch_size = 1  # 你想要的 batch 大小
		batch_images = []

		for _ in range(batch_size):
			if self.idx >= len(self.image_paths):
				break
			image_path = self.image_paths[self.idx]
			input_data = self.preprocess(image_path)
			batch_images.append(input_data[0])  # remove batch dim to stack later
			self.idx += 1
		
		if not batch_images:
			return None

		batch = np.stack(batch_images, axis=0)  # shape: (B, 3, 640, 640)
		print(f"[INFO] Feeding batch, shape = {batch.shape}")
		return {self.input_name: batch}

### Step 2. Static quantize the model.

In [32]:
import os, onnx, datetime
from ultralytics import YOLO
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat

class StaticQuantizer():
	def __init__(self, model_path, data, nodes_exclude, suffix, print_original=False):
		self.model_path = model_path
		self.data = data
		self.nodes_exclude = nodes_exclude
		self.print_original = print_original
		self.suffix = suffix
	
	def static_quantize(self):
		if self.print_original:
			# before quant val
			model = YOLO(self.model_path, task="segment")
			performance = model.val(data=self.data)
			### OUTPUT
			output = ">>> VAL BEFORE", performance.box.map50, performance.box.map, performance.seg.map50, performance.seg.map50, "\n MODEL_NAME", self.model_path, "<<<"
			print(output)
			### help func starts (ignore)
			current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
			with open("data.txt", "a", encoding="utf-8") as f:
				f.write(f"[{current_time}] {output}\n")
				f.write("-" * 80 + "\n")
			### help func ends

		# static quantize (main)
		### help func starts (ignore)
		directory = os.path.dirname(self.model_path)
		filename = os.path.basename(self.model_path)
		name, ext = os.path.splitext(filename)
		new_filename = f"QINT16_{name}_{self.suffix}{ext}"	# whatever easy to read with quantized sign and idx
		### help func ends

		quantized_path = os.path.join("q_models", new_filename)
		calibration_image_paths = ['calibration_data/frame_0000.jpg','calibration_data/frame_0001.jpg','calibration_data/frame_0002.jpg',
						'calibration_data/frame_0003.jpg','calibration_data/frame_0004.jpg','calibration_data/frame_0005.jpg',
						'calibration_data/frame_0006.jpg','calibration_data/frame_0007.jpg','calibration_data/frame_0008.jpg',
						'calibration_data/frame_0009.jpg','calibration_data/frame_0010.jpg']
		calibration_data_reader = ImageCalibrationDataReader(calibration_image_paths)
		quantize_static(
			self.model_path,
			quantized_path,
			weight_type=QuantType.QInt16,
			activation_type=QuantType.QUInt16,
			quant_format=QuantFormat.QDQ,
			nodes_to_quantize=self.nodes_exclude,
			per_channel=False,
			calibration_data_reader=calibration_data_reader,
		) # adjust with experiment settings

		# ### help func starts (ignore)
		# inter_model = onnx.load(quantized_path)
		# graph = inter_model.graph
		# nodes_to_remove = []
		# for node in graph.node:
		# 	if node.op_type in ["QuantizeLinear", "DequantizeLinear"] and "output0" in node.output[0]:
		# 		nodes_to_remove.append(node)
		# for node in nodes_to_remove:
		# 	graph.node.remove(node)
		# for node in graph.node:
		# 	if node.op_type == "Concat": 
		# 		graph.output[0].name = node.output[0]  
		# onnx.save(inter_model, quantized_path)
		# ### help func ends

		# after quant val
		q_model = YOLO(quantized_path, task="segment")
		q_performance = q_model.val(data=self.data)
		### OUTPUT
		q_output = ">>> VAL AFTER", q_performance.box.map50, q_performance.box.map, q_performance.seg.map50, q_performance.seg.map50, "\n MODEL_NAME", quantized_path, "<<<"
		print(q_output)
		### help func starts (ignore)
		q_current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
		with open("data.txt", "a", encoding="utf-8") as f:
			f.write(f"[{q_current_time}] {q_output}\n")
			f.write("-" * 80 + "\n")
		### help func ends

### Step 3. Test using YOLO11 Fish.

In [33]:
if __name__ == "__main__":
	model = "q_models/Q_best_0.onnx"
	data = "D:/app/datasets/fish-combined/data.yaml"
	nodes_exclude = [
    '/model.23/cv2.0/cv2.0.0/conv/Conv',
    '/model.23/cv2.0/cv2.0.0/act/Sigmoid',
    '/model.23/cv2.0/cv2.0.0/act/Mul',
    '/model.23/cv2.0/cv2.0.1/conv/Conv',
    '/model.23/cv2.0/cv2.0.1/act/Sigmoid',
    '/model.23/cv2.0/cv2.0.1/act/Mul',
    '/model.23/cv2.0/cv2.0.2/Conv',
    '/model.23/cv2.1/cv2.1.0/conv/Conv',
    '/model.23/cv2.1/cv2.1.0/act/Sigmoid',
    '/model.23/cv2.1/cv2.1.0/act/Mul',
    '/model.23/cv2.1/cv2.1.1/conv/Conv',
    '/model.23/cv2.1/cv2.1.1/act/Sigmoid',
    '/model.23/cv2.1/cv2.1.1/act/Mul',
    '/model.23/cv2.1/cv2.1.2/Conv',
    '/model.23/cv2.2/cv2.2.0/conv/Conv',
    '/model.23/cv2.2/cv2.2.0/act/Sigmoid',
    '/model.23/cv2.2/cv2.2.0/act/Mul',
    '/model.23/cv2.2/cv2.2.1/conv/Conv',
    '/model.23/cv2.2/cv2.2.1/act/Sigmoid',
    '/model.23/cv2.2/cv2.2.1/act/Mul',
    '/model.23/cv2.2/cv2.2.2/Conv',
    '/model.23/cv4.0/cv4.0.0/conv/Conv',
    '/model.23/cv4.0/cv4.0.0/act/Sigmoid',
    '/model.23/cv4.0/cv4.0.0/act/Mul',
    '/model.23/cv4.0/cv4.0.1/conv/Conv',
    '/model.23/cv4.0/cv4.0.1/act/Sigmoid',
    '/model.23/cv4.0/cv4.0.1/act/Mul',
    '/model.23/cv4.0/cv4.0.2/Conv',
    '/model.23/cv4.1/cv4.1.0/conv/Conv',
    '/model.23/cv4.1/cv4.1.0/act/Sigmoid',
    '/model.23/cv4.1/cv4.1.0/act/Mul',
    '/model.23/cv4.1/cv4.1.1/conv/Conv',
    '/model.23/cv4.1/cv4.1.1/act/Sigmoid',
    '/model.23/cv4.1/cv4.1.1/act/Mul',
    '/model.23/cv4.1/cv4.1.2/Conv',
    '/model.23/cv4.2/cv4.2.0/conv/Conv',
    '/model.23/cv4.2/cv4.2.0/act/Sigmoid',
    '/model.23/cv4.2/cv4.2.0/act/Mul',
    '/model.23/cv4.2/cv4.2.1/conv/Conv',
    '/model.23/cv4.2/cv4.2.1/act/Sigmoid',
    '/model.23/cv4.2/cv4.2.1/act/Mul',
    '/model.23/cv4.2/cv4.2.2/Conv'
]
	idx = 0
	quantizer = StaticQuantizer(model, data, nodes_exclude, idx, print_original=True)
	quantizer.static_quantize()

Ultralytics 8.3.203  Python-3.11.0 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Loading q_models/Q_best_0.onnx for ONNX Runtime inference...
WARNING Failed to start ONNX Runtime with CUDA. Using CPU...
Using ONNX Runtime CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.10.0 ms, read: 82.513.6 MB/s, size: 11.7 KB)
val: Scanning D:\APP\datasets\fish-combined\test\labels.cache... 559 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 559/559 558.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 559/559 14.0it/s 40.0s<0.1s
                   all        559        570      0.842      0.807      0.896      0.729      0.846       0.81      0.898      0.704
                   cod        115        115      0.936      0.896      0.967      0.898      0.936      0.896      0.967      0.871
               

('>>> VAL BEFORE', 0.8963076881429871, 0.7288552645882801, 0.8980341680095046, 0.8980341680095046, '\n MODEL_NAME', 'q_models/Q_best_0.onnx', '<<<')
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)


d:\APP\Coding\python311\Lib\site-packages\onnxruntime\quantization\base_quantizer.py:232: RuntimeWarning: invalid value encountered in cast
  quantized_data = (np.asarray(bias_data) / bias_scale).round().astype(np.int32)


Ultralytics 8.3.203  Python-3.11.0 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Loading q_models\QINT16_Q_best_0_0.onnx for ONNX Runtime inference...
WARNING Failed to start ONNX Runtime with CUDA. Using CPU...
Using ONNX Runtime CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.10.0 ms, read: 64.119.4 MB/s, size: 12.6 KB)
val: Scanning D:\APP\datasets\fish-combined\test\labels.cache... 559 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 559/559 558.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 559/559 13.1it/s 42.5s<0.1s
                   all        559        570       0.84      0.807      0.896      0.725      0.844      0.809      0.897      0.693
                   cod        115        115      0.936      0.887      0.967      0.897      0.936      0.885      0.967      0.869
      